# PCF Ablation Experiment v2 — Orchestration Notebook

Pipeline: **B0 / C1 / C2 (+C3 optional)** × 6 personas × 5 scenarios × 4 turns × 3 runs.

Toàn bộ logic nằm trong `src/`; notebook này chỉ điều phối.

**Chế độ chạy:**
- `DRY_RUN = True` → không gọi API, sinh dữ liệu giả để kiểm tra pipeline end-to-end.
- Chạy thật: đặt `DRY_RUN = False` và cung cấp `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`.

Pipeline **resume-safe**: mỗi response lưu ra file JSON riêng ngay khi có; chạy lại sẽ bỏ qua các record đã tồn tại.

In [ ]:
# === Section 0 — Setup ===
import os, sys
from pathlib import Path

DRY_RUN = True          # False = goi API that
PILOT_MODE = True       # True = chi chay 1 persona, 1 condition de kiem tra (checklist H)

os.environ['DRY_RUN'] = '1' if DRY_RUN else '0'
if not DRY_RUN:
    # Colab: from google.colab import userdata; os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') ...
    assert os.environ.get('OPENAI_API_KEY'), 'Missing OPENAI_API_KEY'
    assert os.environ.get('ANTHROPIC_API_KEY'), 'Missing ANTHROPIC_API_KEY'

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import load_config, utils
ctx = load_config.load_all()
snap = load_config.snapshot_environment(ctx)
print('Experiment:', ctx['experiment']['experiment_name'])
print('Conditions:', ctx['experiment']['conditions'], '| runs:', ctx['experiment']['num_runs'])
print('Personas:', len(ctx['personas']), '| scenarios:', len(ctx['scenarios']))

In [ ]:
# === Section 1 — Unit tests (counterbalance, prompt rendering, C1 vs C2 isolation) ===
import subprocess
r = subprocess.run([sys.executable, str(PROJECT_ROOT / 'tests' / 'test_counterbalance.py')],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, 'Unit tests failed — fix before generating data'

In [ ]:
# === Section 2 — Generate responses ===
from src import generate_responses

if PILOT_MODE:
    n_done, n_skipped = generate_responses.generate_all(ctx, limit_personas=1, limit_conditions=['C1'])
else:
    n_done, n_skipped = generate_responses.generate_all(ctx)

raw_df = generate_responses.collect_raw_csv()
print(f'new={n_done} skipped={n_skipped} total_on_disk={len(raw_df)}')
raw_df.head()

In [ ]:
# === Section 3 — LLM-as-a-Judge ===
from src import judge_responses
judge_responses.judge_all(ctx)
judged_df = judge_responses.collect_judged_csv()
print(len(judged_df), 'judged responses')
judged_df[['record_id', 'condition', 'persona_consistency', 'memory_retention',
           'style_stability', 'context_awareness', 'contradiction_flag', 'error_label']].head()

In [ ]:
# === Section 4 — Human evaluation sample (blinded, stratified) ===
from src import sample_for_human_eval
sample = sample_for_human_eval.make_sample(judged_df, ctx, fraction=0.25)
print('Sheets written to outputs/human_annotations/ (rater1, rater2 + unblinding key)')

In [ ]:
# === Section 5 — Statistical analysis + figures ===
from src import analyze_results
tests, figs = analyze_results.run_full_analysis(judged_df)
print('Figures:', figs)
tests[tests.test == 'wilcoxon'][['metric', 'comparison', 'p_value', 'p_holm', 'effect_size', 'mean_a', 'mean_b']]

In [ ]:
# === Section 6 — Agreement (chay SAU khi 2 rater cham xong) ===
# from src import analyze_results, utils
# agreement = analyze_results.compute_agreement(
#     utils.HUMAN_DIR / 'human_evaluation_sheet_rater1.csv',
#     utils.HUMAN_DIR / 'human_evaluation_sheet_rater2.csv',
#     judged_df)
# agreement